## 単元素からなる基底状態の結晶構造

単元素からなる基底状態の結晶構造に対して原子の説明変数データを用います
データは全てwikipediaからとっています。


**説明変数**

1. min_oxidation_state, max_oxidation_state: min. and max of oxidation state
2. row group
3. s p d f: valence electrion occupation
4. atomic_radius_calculated: calculated atomic radius
5. X(chi), IP, EA : electronegativity, ionization potential, electron affinity


**目的変数**

0. misc (black)
1. hcp (red)
2. bcc (blue)
3. fcc (green)


|1|2|3|4|5|6|7|8|9|10|11|12|13|14|15|16|17|18|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|<font color="red">H</font>| _ | _ |_ |_ |_ |_ |_ |_ |_ |_ |_ |_ |_ |_ |_ |_ |<font color="red">He</font>|
|<font color="blue">Li</font>|<font color="red">Be</font>|_|_|_|_|_|_|_|_|_|_|B|C|N|O|F|<font color="green">Ne</font>|
|<font color="blue">Na</font>|<font color="red">Mg</font>|_|_|_|_|_|_|_|_|_|_|<font color="green">Al</font>|Si|P|S|Cl|<font color="green">Ar</font>|
|<font color="blue">K</font>|<font color="green">Ca</font>|<font color="red">Sc</font>|<font color="red">Ti</font>|<font color="blue">V</font>|<font color="blue">Cr</font>|Mn|<font color="blue">Fe</font>|<font color="red">Co</font>|<font color="green">Ni</font>|<font color="green">Cu</font>|<font color="red">Zn</font>|Ga|Ge|As|Se|Br|<font color="green">Kr</font>|
|<font color="blue">Rb</font>|<font color="green">Sr</font>|<font color="red">Y</font>|<font color="red">Zr</font>|<font color="blue">Nb</font>|<font color="blue">Mo</font>|<font color="red">Tc</font>|<font color="red">Ru</font>|<font color="green">Rh</font>|<font color="green">Pd</font>|<font color="green">Ag</font>|<font color="red">Cd</font>|In|Sn|Sb|Te|I|<font color="green">Xe</font>|
|<font color="blue">Cs</font>|<font color="blue">Ba</font>|_|<font color="red">Hf</font>|<font color="blue">Ta</font>|<font color="blue">W</font>|<font color="red">Re</font>|<font color="red">Os</font>|<font color="green">Ir</font>|<font color="green">Pt</font>|<font color="green">Au</font>|Hg|<font color="red">Tl</font>|<font color="green">Pb</font>|Bi|Po|At|Rn|
|Fr|Ra|_|_|_|_|_|_|_|_|_|_|_|_|_|_|_|_|
|_|_|La|<font color="green">Ce</font>|Pr|Nd|Pm|Sm|<font color="blue">Eu</font>|<font color="red">Gd</font>|<font color="red">Tb</font>|<font color="red">Dy</font>|<font color="red">Ho</font>|<font color="red">Er</font>|<font color="red">Tm</font>|<font color="green">Yb</font>|<font color="red">Lu</font>|_|
|_|_|<font color="green">Ac</font>|<font color="green">Th</font>|Pa|U|Np|Pu|Am|Cm|Bk|Cf|Es|Fm|Md|No|Lr|_|


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("../data/mono_structure.csv")
descriptor_names = [ 'min_oxidation_state', 'max_oxidation_state', 'row',
       'group', 's', 'p', 'd', 'f', 'atomic_radius_calculated', 'X', 'IP',
       'EA'] 
target_name = 'crystal_structure'
df.columns

In [ ]:
df

In [ ]:
def show_labels(df, target_name, target_classes):
    fig, ax = plt.subplots()
    df.hist(target_name, ax=ax)
    ax.set_xticks(list(target_classes.keys()))
    ax.set_xticklabels(list(target_classes.values()))
    
target_classes = {0:"misc",1:"hcp",2:"bcc",3:"fcc"}
show_labels(df, target_name, target_classes)

In [ ]:
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist,squareform
from scipy.cluster.hierarchy import dendrogram, linkage
import copy
from scipy.stats import pearsonr
import numpy as np

def make_linkage(df, descriptor_names, target_name, corr="minus_abs_pearson"):
    labels = copy.deepcopy(descriptor_names)
    labels.append(target_name)
    Xraw = df.loc[:,labels].values
    scaler = StandardScaler()
    X = scaler.fit_transform(Xraw)
    df_tmp = pd.DataFrame(X)
    if corr=="minus_abs_pearson":
        corr = 1- np.abs(df_tmp.corr())
    else:
        raise ValueError("unknown corr={}".format(corr))
    pairdistance = squareform(corr)
    Z = linkage(pairdistance)
    return Z, labels

def show_dendrogram(Z,labels, corr):
    fig, ax = plt.subplots()
    dendrogram(Z,labels=labels,orientation="left", ax=ax)
    ax.set_xlabel(corr)
    fig.tight_layout()
    fig.show()
    
corr="minus_abs_pearson"
Z, labels = make_linkage(df, descriptor_names, target_name, corr)
show_dendrogram(Z,labels, corr)

In [ ]:
import os
import seaborn as sns
from copy import deepcopy

IMAGE_DIR = "image_keep"

imgfile = os.path.join(IMAGE_DIR, "monostructure_pairplot.png")
if not os.path.isfile(imgfile):
    alllabels = deepcopy(descriptor_names)
    alllabels.append(target_name)
    img = sns.pairplot(df[alllabels])
    os.makedirs(IMAGE_DIR, exist_ok=True)
    img.savefig(imgfile)
    
from IPython import display
display.Image(imgfile)

参考文献

1. wikipedia, 例えばFeに関して
https://en.wikipedia.org/wiki/Iron
